# Stage 8 large LLM JSON-validator baseline

Run newer local Hugging Face models through the strict Vega-Lite JSON validator with up to three validator-feedback retries.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit

from google.colab import drive

try:
    from google.colab import userdata
except Exception:
    userdata = None

DRIVE_ROOT = Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part')
REPO_ROOT = Path('/content/petukhov_t2v_repo')
REPO_URL = 'https://github.com/petrussia/NL2BI-AI-assistant.git'
BRANCH = 'experiments/peter'
TOKEN_FILE = DRIVE_ROOT / 'secrets' / 'github_token.txt'
HF_TOKEN_FILE = DRIVE_ROOT / 'secrets' / 'hf_token.txt'


def read_secret(name: str, fallback: Path | None = None) -> str | None:
    if userdata is not None:
        try:
            token = userdata.get(name)
            if token:
                return token.strip()
        except Exception:
            pass
    if fallback and fallback.exists():
        return fallback.read_text(encoding='utf-8').strip()
    return None


def auth_url(token: str | None) -> str:
    if not token:
        return REPO_URL
    parts = urlsplit(REPO_URL)
    return urlunsplit((parts.scheme, f'x-access-token:{quote(token, safe="")}@{parts.netloc}', parts.path, parts.query, parts.fragment))


def run(cmd: list[str], cwd: Path | None = None) -> None:
    shown = ['***' if 'x-access-token:' in part else part for part in cmd]
    print('RUN', ' '.join(shown))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)


drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['T2V_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['T2V_REPO_ROOT'] = str(REPO_ROOT)

repo_url = auth_url(read_secret('GITHUB_TOKEN', TOKEN_FILE))
if not REPO_ROOT.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', repo_url, str(REPO_ROOT)])
else:
    run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_ROOT)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', BRANCH], cwd=REPO_ROOT)
    try:
        run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_ROOT)
    except subprocess.CalledProcessError:
        print('git pull --ff-only failed; resetting Colab checkout to origin branch')
        run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_ROOT)
run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_ROOT)

hf_token = read_secret('HF_TOKEN', HF_TOKEN_FILE)
if hf_token:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)])
run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-U',
    'transformers>=4.56',
    'accelerate>=1.0',
    'bitsandbytes',
    'sentencepiece',
    'safetensors',
    'protobuf',
    'compressed-tensors>=0.10',
    'mistral-common>=1.5.4',
])
run([sys.executable, '-c', 'import t2v_eval; print(t2v_eval.__version__)'])
print('STAGE8_SETUP_OK')


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path(os.environ['T2V_REPO_ROOT'])
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_stage8_large_llm.py'),
    '--list-models',
]
print('RUN', ' '.join(cmd))
subprocess.check_call(cmd)

try:
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(device)
        gpu = {
            'name': torch.cuda.get_device_name(device),
            'total_vram_gb': round(props.total_memory / (1024 ** 3), 2),
        }
    else:
        gpu = {'cuda_available': False}
except Exception as exc:
    gpu = {'error': str(exc)}
print(json.dumps({'gpu': gpu}, ensure_ascii=False, indent=2))
print('STAGE8_MODELS_OK')


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

MODEL_KEY = 'gemma4_e2b_it'
MARKER = 'STAGE8_GEMMA4_E2B_IT_OK'
SAMPLE_SIZE = os.environ.get('STAGE8_SAMPLE_SIZE', '20')
RENDER_LIMIT = os.environ.get('STAGE8_RENDER_LIMIT', '0')

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
examples = drive_root / 'datasets' / 'processed' / 'nvbench_postquery' / 'examples_sample200.jsonl'
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_stage8_large_llm.py'),
    '--model-key', MODEL_KEY,
    '--examples', str(examples),
    '--drive-root', str(drive_root),
    '--sample-size', SAMPLE_SIZE,
    '--render-limit', RENDER_LIMIT,
    '--json',
]
if os.environ.get('STAGE8_ALLOW_LOW_VRAM') == '1':
    cmd.append('--allow-low-vram')
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Stage 8 {MODEL_KEY} failed: {result.returncode}')
print(MARKER)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

MODEL_KEY = 'qwen3_14b'
MARKER = 'STAGE8_QWEN3_14B_OK'
SAMPLE_SIZE = os.environ.get('STAGE8_SAMPLE_SIZE', '20')
RENDER_LIMIT = os.environ.get('STAGE8_RENDER_LIMIT', '0')

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
examples = drive_root / 'datasets' / 'processed' / 'nvbench_postquery' / 'examples_sample200.jsonl'
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_stage8_large_llm.py'),
    '--model-key', MODEL_KEY,
    '--examples', str(examples),
    '--drive-root', str(drive_root),
    '--sample-size', SAMPLE_SIZE,
    '--render-limit', RENDER_LIMIT,
    '--json',
]
if os.environ.get('STAGE8_ALLOW_LOW_VRAM') == '1':
    cmd.append('--allow-low-vram')
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Stage 8 {MODEL_KEY} failed: {result.returncode}')
print(MARKER)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

MODEL_KEY = 'gemma3_12b_it'
MARKER = 'STAGE8_GEMMA3_12B_IT_OK'
SAMPLE_SIZE = os.environ.get('STAGE8_SAMPLE_SIZE', '20')
RENDER_LIMIT = os.environ.get('STAGE8_RENDER_LIMIT', '0')

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
examples = drive_root / 'datasets' / 'processed' / 'nvbench_postquery' / 'examples_sample200.jsonl'
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_stage8_large_llm.py'),
    '--model-key', MODEL_KEY,
    '--examples', str(examples),
    '--drive-root', str(drive_root),
    '--sample-size', SAMPLE_SIZE,
    '--render-limit', RENDER_LIMIT,
    '--json',
]
if os.environ.get('STAGE8_ALLOW_LOW_VRAM') == '1':
    cmd.append('--allow-low-vram')
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Stage 8 {MODEL_KEY} failed: {result.returncode}')
print(MARKER)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

MODEL_KEY = 'mistral_small_32_24b_bnb4'
MARKER = 'STAGE8_MISTRAL_SMALL_32_BNB4_OK'
SAMPLE_SIZE = os.environ.get('STAGE8_SAMPLE_SIZE', '20')
RENDER_LIMIT = os.environ.get('STAGE8_RENDER_LIMIT', '0')

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
examples = drive_root / 'datasets' / 'processed' / 'nvbench_postquery' / 'examples_sample200.jsonl'
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_stage8_large_llm.py'),
    '--model-key', MODEL_KEY,
    '--examples', str(examples),
    '--drive-root', str(drive_root),
    '--sample-size', SAMPLE_SIZE,
    '--render-limit', RENDER_LIMIT,
    '--json',
]
if os.environ.get('STAGE8_ALLOW_LOW_VRAM') == '1':
    cmd.append('--allow-low-vram')
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Stage 8 {MODEL_KEY} failed: {result.returncode}')
print(MARKER)


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

repo_root = Path(os.environ['T2V_REPO_ROOT'])
drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
config = json.loads((repo_root / 'configs' / 'stage8_large_llm_models.json').read_text(encoding='utf-8'))
rows = []
for key, model in config['models'].items():
    run_id = model['run_id']
    run_root = drive_root / 'runs' / run_id
    method = model['method_name']
    metrics_path = run_root / 'metrics' / method / 'aggregate_metrics.csv'
    summary_path = run_root / 'stage8_model_summary.json'
    row = {
        'model_key': key,
        'run_id': run_id,
        'method': method,
        'exists': run_root.exists(),
        'summary': str(summary_path) if summary_path.exists() else None,
    }
    if metrics_path.exists():
        metrics = pd.read_csv(metrics_path).iloc[0].to_dict()
        for metric in ['chart_type_accuracy', 'field_selection_f1', 'normalized_exact_match', 'vega_lite_validity', 'failure_rate', 'latency_ms']:
            row[metric] = metrics.get(metric)
    rows.append(row)
print(json.dumps(rows, ensure_ascii=False, indent=2))
print('STAGE8_SUMMARY_OK')
